# Task for Today  

***

## EEG Emotion Prediction  

Given *EEG data from subjects who were watching movies*, let's try to predict the **emotional state** of a subject during a given movie.  
  
We will use a TensorFlow recurrent neural network to make our predictions.

# Getting Started

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import tensorflow as tf

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
data = pd.read_csv('../input/eeg-brainwave-dataset-feeling-emotions/emotions.csv')

In [ ]:
data

In [ ]:
sample = data.loc[0, 'fft_0_b':'fft_749_b']

plt.figure(figsize=(16, 10))
plt.plot(range(len(sample)), sample)
plt.title("Features fft_0_b through fft_749_b")
plt.show()

In [ ]:
data['label'].value_counts()

In [ ]:
label_mapping = {'NEGATIVE': 0, 'NEUTRAL': 1, 'POSITIVE': 2}

# Preprocessing

In [ ]:
def preprocess_inputs(df):
    df = df.copy()
    
    df['label'] = df['label'].replace(label_mapping)
    
    y = df['label'].copy()
    X = df.drop('label', axis=1).copy()
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=123)
    
    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = preprocess_inputs(data)

In [ ]:
X_train

# Modeling

In [ ]:
inputs = tf.keras.Input(shape=(X_train.shape[1],))

expand_dims = tf.expand_dims(inputs, axis=2) # klasik ann için yeterli ancak gru için zaman boyutu ekleniyor(3. boyut zaman)

gru = tf.keras.layers.GRU(256, return_sequences=True)(expand_dims) # 256=modelin öğrenme kapasitesi(hücre sayısı), büyürse overfitting olabilir

flatten = tf.keras.layers.Flatten()(gru) # 3 boyutlu vektörü tek boyutlu vektöre çeviriyor.

outputs = tf.keras.layers.Dense(3, activation='softmax')(flatten) # çıktı katmanı, 3 sınıflı, softmax=olasılık dağıtır en yükseği seçer


model = tf.keras.Model(inputs=inputs, outputs=outputs)
print(model.summary())

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping( # eeg verilerinde overfitting çok yaygın olduğu için 5 tur boyunca hiç eğitim olmazsa dur
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)

# Results

In [ ]:
model_acc = model.evaluate(X_test, y_test, verbose=0)[1]
print("Test Accuracy: {:.3f}%".format(model_acc * 100))

In [ ]:
y_pred = np.array(list(map(lambda x: np.argmax(x), model.predict(X_test))))

cm = confusion_matrix(y_test, y_pred)
clr = classification_report(y_test, y_pred, target_names=label_mapping.keys())

plt.figure(figsize=(8, 8))
sns.heatmap(cm, annot=True, vmin=0, fmt='g', cbar=False, cmap='Blues')
plt.xticks(np.arange(3) + 0.5, label_mapping.keys())
plt.yticks(np.arange(3) + 0.5, label_mapping.keys())
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

print("Classification Report:\n----------------------\n", clr)

In [ ]:
inputs = tf.keras.Input(shape=(X_train.shape[1],))

# Zaman boyutu ekleniyor
expand_dims = tf.expand_dims(inputs, axis=2)

# LSTM katmanı
lstm = tf.keras.layers.LSTM(
    256,
    return_sequences=True
)(expand_dims)

# Düzleştirme
flatten = tf.keras.layers.Flatten()(lstm)

# Çıkış katmanı
outputs = tf.keras.layers.Dense(3, activation='softmax')(flatten)

# Model
lstm_model = tf.keras.Model(inputs=inputs, outputs=outputs)
print(lstm_model.summary())


In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)


In [ ]:
def plot_history(hist, title):
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(hist.history['accuracy'], label='Train')
    plt.plot(hist.history['val_accuracy'], label='Validation')
    plt.title(f'{title} - Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(hist.history['loss'], label='Train')
    plt.plot(hist.history['val_loss'], label='Validation')
    plt.title(f'{title} - Loss')
    plt.legend()

    plt.show()


plot_history(history, "GRU")
plot_history(lstm_history, "LSTM")


In [ ]:
inputs = tf.keras.Input(shape=(X_train.shape[1],))

# Zaman boyutu ekleniyor
expand_dims = tf.expand_dims(inputs, axis=2)

# Simple RNN katmanı
lstm = tf.keras.layers.SimpleRNN(
    256,
    return_sequences=True
)(expand_dims)

# Düzleştirme
flatten = tf.keras.layers.Flatten()(lstm)

# Çıkış katmanı
outputs = tf.keras.layers.Dense(3, activation='softmax')(flatten)

# Model
simple_rnn_model = tf.keras.Model(inputs=inputs, outputs=outputs)
print(simple_rnn_model.summary())


In [ ]:
simple_rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

simple_rnn_history = simple_rnn_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=50,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True
        )
    ]
)


In [ ]:
gru_acc = model.evaluate(X_test, y_test, verbose=0)[1]
lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)[1]
simple_rnn_acc = simple_rnn_model.evaluate(X_test, y_test, verbose=0)[1]

print(f"GRU Accuracy: {gru_acc}")
print(f"LSTM Accuracy: {lstm_acc}")
print(f"Simple RNN Accuracy: {simple_rnn_acc}")

In [ ]:
y_pred_gru = np.argmax(model.predict(X_test), axis=1)
cm_gru = confusion_matrix(y_test, y_pred_gru)

y_pred_lstm = np.argmax(lstm_model.predict(X_test), axis=1)
cm_lstm = confusion_matrix(y_test, y_pred_lstm)

y_pred_simple_rnn = np.argmax(simple_rnn_model.predict(X_test), axis=1)
cm_simple_rnn = confusion_matrix(y_test, y_pred_simple_rnn)

print("GRU Matrix:")
print(cm_gru)
print("LSTM Matrix:")
print(cm_lstm)
print("Simple RNN Matrix:")
print(cm_simple_rnn)

In [ ]:
clr_gru = classification_report(y_test, y_pred_gru, target_names=label_mapping.keys())

clr_lstm = classification_report(y_test, y_pred_lstm, target_names=label_mapping.keys())

clr_simple_rnn = classification_report(y_test, y_pred_simple_rnn, target_names=label_mapping.keys())

print("GRU Classification Report:")
print(clr_gru)
print("LSTM Classification Report:")
print(clr_lstm)
print("Simple RNN Classification Report:")
print(clr_simple_rnn)

In [ ]:
def plot_history(hist, title):
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(hist.history['accuracy'], label='Train')
    plt.plot(hist.history['val_accuracy'], label='Validation')
    plt.title(f'{title} - Accuracy')
    plt.legend()

    plt.subplot(1,2,2)
    plt.plot(hist.history['loss'], label='Train')
    plt.plot(hist.history['val_loss'], label='Validation')
    plt.title(f'{title} - Loss')
    plt.legend()

    plt.show()


plot_history(history, "GRU")
plot_history(lstm_history, "LSTM")
plot_history(simple_rnn_history,"Simple RNN")


In [ ]:
lstm_model.save('/kaggle/working/lstm_model.keras')